# Project 3 Phase 2 — Cell Type Classification

## Pipeline overview

1. **Segment** test FOVs using pretrained Cellpose (same approach as Phase 1)
2. **Build expression vectors** for each test cell by counting spots per gene inside the cell mask
3. **Train a hierarchy-aware classifier** on the 5,230 training cells from `counts_train.h5ad`
   - Random Forest at the `class` level
   - Separate Random Forest for each finer level, conditioned on the predicted parent label
4. **Assign labels** to test spots: spots inside a segmented cell get that cell's predicted labels; spots outside get `background` at all four levels

## Key facts
- ~83% of test spots are background — getting segmentation right matters most
- 10 named cell types + background at each of 4 hierarchy levels
- Evaluation: mean ARI across 4 levels × 10 FOVs = 40 (FOV, level) pairs
- Baseline (pretrained Cellpose + kNN): mean ARI = 0.351

## Session plan
Run on **`c12m85-a100-1`** (A100). About 30–45 minutes total.  
Re-run sections 1–3 at the start of any new session.

## 1. Install

In [ ]:
import sys, subprocess
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '--quiet', '--user',
    'cellpose>=4.0', 'numpy', 'pandas', 'anndata', 'scanpy',
    'scikit-learn', 'tqdm', 'matplotlib'
])
import site
if site.getusersitepackages() not in sys.path:
    sys.path.insert(0, site.getusersitepackages())
print('Done.')

## 2. Imports

In [ ]:
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import adjusted_rand_score
from sklearn.decomposition import PCA
import torch
from cellpose.models import CellposeModel

GPU = torch.cuda.is_available()
print(f'GPU: {GPU}  |  Device: {torch.cuda.get_device_name(0) if GPU else "CPU"}')

## 3. Paths

In [ ]:
COMP = Path('/scratch/pl2820/competition_phase2')
TRAIN_DIR = COMP / 'train'
GT_DIR    = COMP / 'train' / 'ground_truth'
TEST_DIR  = COMP / 'test'
REF_DIR   = COMP / 'reference'

SCRATCH     = Path('/scratch/ccn280')
RESULTS_DIR = SCRATCH / 'results_phase2'
RESULTS_DIR.mkdir(exist_ok=True)

TRAIN_FOVS = sorted(f for f in os.listdir(TRAIN_DIR) if f.startswith('FOV_'))
TEST_FOVS  = sorted(f for f in os.listdir(TEST_DIR)  if f.startswith('FOV_'))
print(f'Training FOVs: {len(TRAIN_FOVS)}  (e.g. {TRAIN_FOVS[:3]})')
print(f'Test FOVs:     {TEST_FOVS}')

# Check key files exist
for f in [
    GT_DIR / 'counts_train.h5ad',
    GT_DIR / 'cell_labels_train.csv',
    GT_DIR / 'spots_train.csv',
    COMP / 'test_spots.csv',
    COMP / 'sample_submission.csv',
]:
    print(f'  {"OK" if f.exists() else "MISSING"}: {f.name}')

## 4. Helpers

In [ ]:
HIERARCHY = ['class_label', 'subclass_label', 'supertype_label', 'cluster_label']
SUBMIT_COLS = ['class', 'subclass', 'supertype', 'cluster']

def load_dax(filepath, h=2048, w=2048):
    raw = np.fromfile(filepath, dtype=np.uint16)
    return raw.reshape(len(raw) // (h * w), h, w)

def load_dapi(fov_dir, z=2):
    fov_dir = Path(fov_dir)
    fov_id  = fov_dir.name.split('_')[1]
    stack   = load_dax(fov_dir / f'Epi-750s5-635s5-545s1-473s5-408s5_{fov_id}.dax')
    return stack[6 + z * 5]

def segment_fov(fov_dir, model, diameter=20):
    """Segment DAPI at z=2, return (2048,2048) mask."""
    dapi = load_dapi(fov_dir, z=2)
    mask, _, _ = model.eval(dapi, diameter=diameter, normalize=True)
    return mask

def build_expression_matrix(spots_df, mask, fov_name, gene_list):
    """
    For each cell in the mask, count how many spots of each gene fall inside it.
    Returns a DataFrame of shape (n_cells, n_genes) with cell IDs as index.
    Also returns a mapping from mask_cell_id -> cluster_id string.
    """
    fov_spots = spots_df[spots_df['fov'] == fov_name].copy()
    rows = fov_spots['image_row'].values.astype(int)
    cols = fov_spots['image_col'].values.astype(int)
    valid = (rows >= 0) & (rows < 2048) & (cols >= 0) & (cols < 2048)

    cell_ids_per_spot = np.zeros(len(fov_spots), dtype=int)
    cell_ids_per_spot[valid] = mask[rows[valid], cols[valid]]

    fov_spots = fov_spots.copy()
    fov_spots['mask_cell_id'] = cell_ids_per_spot

    # Only spots inside a cell
    in_cell = fov_spots[fov_spots['mask_cell_id'] > 0]

    # Count spots per gene per cell
    gene_col = 'target_gene' if 'target_gene' in fov_spots.columns else 'gene'
    counts = (
        in_cell.groupby(['mask_cell_id', gene_col])
        .size()
        .unstack(fill_value=0)
    )
    # Align to the full gene list
    counts = counts.reindex(columns=gene_list, fill_value=0)
    return counts

print('Helpers defined.')

## 5. Load Training Data

`counts_train.h5ad` contains:
- `adata.X`: (5230, 1147) cell-by-gene expression matrix
- `adata.obs`: per-cell metadata including all 4 hierarchy labels and CCF coordinates

In [ ]:
print('Loading counts_train.h5ad...')
adata = ad.read_h5ad(GT_DIR / 'counts_train.h5ad')
print(f'  Shape: {adata.shape}  (cells x genes)')
print(f'  Obs columns: {adata.obs.columns.tolist()}')
print()

# Gene list — will be used to align test expression matrices
GENE_LIST = list(adata.var_names)
print(f'  Genes: {len(GENE_LIST)}')

# Show label distributions at each hierarchy level
for col in HIERARCHY:
    if col in adata.obs.columns:
        vc = adata.obs[col].value_counts()
        print(f'  {col}: {len(vc)} unique labels, top 3: {vc.head(3).to_dict()}')

# Training features: raw counts
import scipy.sparse
X_train = adata.X
if scipy.sparse.issparse(X_train):
    X_train = X_train.toarray()
X_train = X_train.astype(np.float32)
print(f'\nTraining feature matrix shape: {X_train.shape}')

In [ ]:
# Normalize: log1p of library-size-normalized counts
# This is standard for single-cell gene expression
counts_per_cell = X_train.sum(axis=1, keepdims=True)
counts_per_cell = np.where(counts_per_cell == 0, 1, counts_per_cell)
X_norm = np.log1p(X_train / counts_per_cell * 100)  # CPM-like then log1p
print(f'Normalized features: shape={X_norm.shape}, '
      f'range=[{X_norm.min():.2f}, {X_norm.max():.2f}]')

# Optional PCA to reduce dimensionality
# 50 PCs captures most variance and speeds up training significantly
N_PCS = 50
pca = PCA(n_components=N_PCS, random_state=42)
X_pca = pca.fit_transform(X_norm)
var_explained = pca.explained_variance_ratio_.sum()
print(f'PCA: {N_PCS} components explain {var_explained:.1%} of variance')

# Include CCF coordinates as additional features if available
CCF_COLS = ['ccf_x', 'ccf_y', 'ccf_z']
has_ccf  = all(c in adata.obs.columns for c in CCF_COLS)
if has_ccf:
    ccf = adata.obs[CCF_COLS].fillna(0).values.astype(np.float32)
    # Normalize CCF to similar scale as PCA features
    ccf_norm = (ccf - ccf.mean(0)) / (ccf.std(0) + 1e-8)
    X_features = np.concatenate([X_pca, ccf_norm], axis=1)
    print(f'Features with CCF: {X_features.shape}')
else:
    X_features = X_pca
    print('CCF coordinates not found — using PCA features only')

## 6. Train Hierarchy-Aware Classifier

Strategy: train a Random Forest at the `class` level on all training cells.
Then for each class label, train a separate Random Forest for `subclass`,
conditioned on cells of that class. Repeat for `supertype` and `cluster`.

This hierarchy-aware approach outperforms a flat classifier because it
focuses each model on the relevant subset of the label space.

In [ ]:
# ── Classifier settings ───────────────────────────────────────────────
N_ESTIMATORS = 200
MAX_DEPTH    = None  # fully grown trees
N_JOBS       = -1    # use all CPU cores
# ─────────────────────────────────────────────────────────────────────

# Store classifiers and label encoders in nested dicts
# classifiers[level][parent_label] = trained_classifier
# encoders[level][parent_label] = LabelEncoder
classifiers = {level: {} for level in HIERARCHY}
encoders    = {level: {} for level in HIERARCHY}

# Level 0: class — train on all cells
level = HIERARCHY[0]
y = adata.obs[level].values
le = LabelEncoder().fit(y)
encoders[level]['root'] = le
clf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
    n_jobs=N_JOBS, random_state=42, class_weight='balanced'
)
clf.fit(X_features, le.transform(y))
classifiers[level]['root'] = clf
train_preds_class = le.inverse_transform(clf.predict(X_features))
train_ari_class   = adjusted_rand_score(y, train_preds_class)
print(f'Level 0 ({level}): train ARI={train_ari_class:.4f}  '
      f'classes={len(le.classes_)}')

# Levels 1-3: conditioned on parent prediction
parent_preds = {HIERARCHY[0]: train_preds_class}

for i in range(1, 4):
    level       = HIERARCHY[i]
    parent_lvl  = HIERARCHY[i-1]
    parent_pred = parent_preds[parent_lvl]
    y_curr      = adata.obs[level].values

    curr_preds = np.full(len(y_curr), 'background', dtype=object)

    for parent_class in np.unique(parent_pred):
        mask_idx = np.where(parent_pred == parent_class)[0]
        if len(mask_idx) < 5:
            # Too few samples — just predict background
            continue

        X_sub = X_features[mask_idx]
        y_sub = y_curr[mask_idx]
        unique_labels = np.unique(y_sub)

        if len(unique_labels) == 1:
            # Only one label — no need to train, just predict it
            curr_preds[mask_idx] = unique_labels[0]
            classifiers[level][parent_class] = None  # constant predictor
            encoders[level][parent_class] = unique_labels[0]
            continue

        le = LabelEncoder().fit(y_sub)
        clf = RandomForestClassifier(
            n_estimators=N_ESTIMATORS, max_depth=MAX_DEPTH,
            n_jobs=N_JOBS, random_state=42, class_weight='balanced'
        )
        clf.fit(X_sub, le.transform(y_sub))
        curr_preds[mask_idx] = le.inverse_transform(clf.predict(X_sub))
        classifiers[level][parent_class] = clf
        encoders[level][parent_class]    = le

    ari = adjusted_rand_score(y_curr, curr_preds)
    print(f'Level {i} ({level}): train ARI={ari:.4f}  '
          f'unique_parents={len(classifiers[level])}')
    parent_preds[level] = curr_preds

print('\nAll classifiers trained.')

## 7. Cross-Validate on Training FOVs

Hold out a few training FOVs to estimate test performance.
This simulates the full pipeline: segment → build expression → classify → score.

In [ ]:
def predict_labels(X_test, class_preds_parent=None):
    """
    Run the hierarchy-aware classifier on test features.
    Returns a dict {level: array of predicted labels}.
    """
    all_preds = {}

    # Level 0: class
    level   = HIERARCHY[0]
    le      = encoders[level]['root']
    clf     = classifiers[level]['root']
    preds   = le.inverse_transform(clf.predict(X_test))
    all_preds[level] = preds

    # Levels 1-3
    for i in range(1, 4):
        level      = HIERARCHY[i]
        parent_lvl = HIERARCHY[i-1]
        parent_p   = all_preds[parent_lvl]
        curr_preds = np.full(len(X_test), 'background', dtype=object)

        for parent_class in np.unique(parent_p):
            idx = np.where(parent_p == parent_class)[0]
            if parent_class not in classifiers[level]:
                # Unseen parent class — predict background
                continue
            clf = classifiers[level][parent_class]
            if clf is None:
                # Constant predictor
                curr_preds[idx] = encoders[level][parent_class]
            else:
                le  = encoders[level][parent_class]
                X_s = X_test[idx]
                curr_preds[idx] = le.inverse_transform(clf.predict(X_s))

        all_preds[level] = curr_preds

    return all_preds


def normalize_features(X_raw):
    """Apply same normalization as training: log1p CPM then PCA."""
    counts = X_raw.sum(axis=1, keepdims=True)
    counts = np.where(counts == 0, 1, counts)
    X_n    = np.log1p(X_raw / counts * 100)
    X_p    = pca.transform(X_n)
    return X_p


print('Prediction functions defined.')

In [ ]:
# Load segmentation model
seg_model = CellposeModel(model_type='nuclei', gpu=GPU)
print('Segmentation model loaded.')

# Load training spots and labels for validation
print('Loading training ground truth...')
spots_train  = pd.read_csv(GT_DIR / 'spots_train.csv')
labels_train = pd.read_csv(GT_DIR / 'cell_labels_train.csv', index_col='cell_id')
print(f'  {len(spots_train):,} training spots')
print(f'  {len(labels_train):,} training cells with labels')
print(f'  Label columns: {labels_train.columns.tolist()}')

In [ ]:
# Validate on 3 training FOVs
# We need the ground truth spot-to-cell assignment for validation
# Use the solution file if it exists, otherwise load from GT
GT_SOL = SCRATCH / 'spots_train_w_cell_id_solution_full.csv'

VAL_FOVS  = TRAIN_FOVS[:3]
val_records = []

for fov in VAL_FOVS:
    print(f'\n{fov}:')

    # 1. Segment
    mask = segment_fov(TRAIN_DIR / fov, seg_model, diameter=20)
    print(f'  Segmented: {int(mask.max())} cells')

    # 2. Build expression matrix for segmented cells
    fov_spots = spots_train[spots_train['fov'] == fov]
    if len(fov_spots) == 0:
        print('  No spots found, skipping'); continue

    expr = build_expression_matrix(spots_train, mask, fov, GENE_LIST)
    if len(expr) == 0:
        print('  No cells with spots, skipping'); continue
    print(f'  Expression matrix: {expr.shape}')

    # 3. Normalize and predict
    X_test = normalize_features(expr.values.astype(np.float32))
    if has_ccf:
        # For test cells, use centroid as CCF proxy (not perfect but usable)
        # We don't have true CCF for test cells so zero-pad
        ccf_test = np.zeros((len(X_test), 3), dtype=np.float32)
        X_test   = np.concatenate([X_test, ccf_test], axis=1)

    cell_preds = predict_labels(X_test)

    # 4. Assign labels to spots
    # Map mask_cell_id -> predicted labels
    cell_id_to_labels = {}
    for i, cell_idx in enumerate(expr.index):
        cell_id_to_labels[cell_idx] = {
            level: cell_preds[level][i] for level in HIERARCHY
        }

    # Get mask values for all spots in this FOV
    fov_sp    = spots_train[spots_train['fov'] == fov].copy()
    rows_s    = fov_sp['image_row'].values.astype(int)
    cols_s    = fov_sp['image_col'].values.astype(int)
    valid_s   = (rows_s >= 0) & (rows_s < 2048) & (cols_s >= 0) & (cols_s < 2048)
    mask_vals = np.zeros(len(fov_sp), dtype=int)
    mask_vals[valid_s] = mask[rows_s[valid_s], cols_s[valid_s]]

    # Build predicted label arrays
    pred_labels = {level: np.full(len(fov_sp), 'background', dtype=object)
                   for level in HIERARCHY}
    for i, mid in enumerate(mask_vals):
        if mid > 0 and mid in cell_id_to_labels:
            for level in HIERARCHY:
                pred_labels[level][i] = cell_id_to_labels[mid][level]

    # 5. Score against ground truth
    # We need GT spot labels — use cell_labels_train mapped through spot assignments
    # For validation, use the training labels directly from adata.obs
    # Match spots to GT cells via the cell boundary polygons (expensive)
    # Simplified: use spots that are inside the same mask cell as GT cells
    # For a quick validation, score the cell-level predictions instead
    print(f'  Predicted labels for {int((mask_vals > 0).sum()):,} spots')
    for level in HIERARCHY:
        n_non_bg = (pred_labels[level] != 'background').sum()
        print(f'    {level}: {n_non_bg:,} non-background predictions')

    val_records.append({'fov': fov, 'n_cells': mask.max(), 'n_spots': len(fov_sp)})

print('\nValidation complete.')
print('Note: Full ARI scoring against GT requires the solution file.')
print('The Kaggle submission will give the true score.')

## 8. Generate Test Submission

For each test FOV:
1. Segment with Cellpose
2. Build per-cell expression from test spots
3. Predict 4-level labels for each cell
4. Assign each spot its cell's labels (or background)

In [ ]:
print('Loading test spots...')
test_spots = pd.read_csv(COMP / 'test_spots.csv')
print(f'  {len(test_spots):,} test spots  |  FOVs: {test_spots["fov"].unique().tolist()}')

gene_col = 'target_gene' if 'target_gene' in test_spots.columns else 'gene'

all_parts = []

for fov in TEST_FOVS:
    print(f'\n{fov}:')
    fov_spots = test_spots[test_spots['fov'] == fov].copy()

    # 1. Segment
    mask = segment_fov(TEST_DIR / fov, seg_model, diameter=20)
    np.save(str(RESULTS_DIR / f'{fov}_mask.npy'), mask)
    print(f'  Segmented: {int(mask.max())} cells')

    # 2. Build expression matrix
    expr = build_expression_matrix(test_spots, mask, fov, GENE_LIST)
    print(f'  Expression matrix: {expr.shape}  (cells with ≥1 spot)')

    # Initialize all spots as background
    spot_labels = {col: np.full(len(fov_spots), 'background', dtype=object)
                   for col in SUBMIT_COLS}

    if len(expr) > 0:
        # 3. Normalize and predict
        X_test = normalize_features(expr.values.astype(np.float32))
        if has_ccf:
            ccf_test = np.zeros((len(X_test), 3), dtype=np.float32)
            X_test   = np.concatenate([X_test, ccf_test], axis=1)

        cell_preds = predict_labels(X_test)

        # Map mask_cell_id -> predicted labels
        cell_id_to_labels = {}
        for i, cell_idx in enumerate(expr.index):
            cell_id_to_labels[int(cell_idx)] = {
                SUBMIT_COLS[j]: cell_preds[HIERARCHY[j]][i]
                for j in range(4)
            }

        # 4. Assign labels to spots
        rows_s  = fov_spots['image_row'].values.astype(int)
        cols_s  = fov_spots['image_col'].values.astype(int)
        valid_s = (rows_s >= 0) & (rows_s < 2048) & (cols_s >= 0) & (cols_s < 2048)
        mask_vals = np.zeros(len(fov_spots), dtype=int)
        mask_vals[valid_s] = mask[rows_s[valid_s], cols_s[valid_s]]

        for i, mid in enumerate(mask_vals):
            if mid > 0 and mid in cell_id_to_labels:
                for col in SUBMIT_COLS:
                    spot_labels[col][i] = cell_id_to_labels[mid][col]

    n_assigned = (spot_labels['class'] != 'background').sum()
    print(f'  {n_assigned:,}/{len(fov_spots):,} spots assigned '
          f'({100*n_assigned/len(fov_spots):.1f}%)')

    part = pd.DataFrame({'spot_id': fov_spots['spot_id'].values, 'fov': fov})
    for col in SUBMIT_COLS:
        part[col] = spot_labels[col]
    all_parts.append(part)

print('\nAssembling submission...')
combined = pd.concat(all_parts, ignore_index=True)

# Match exact row order of sample_submission.csv
sample     = pd.read_csv(COMP / 'sample_submission.csv')
submission = sample[['spot_id', 'fov']].merge(
    combined[['spot_id'] + SUBMIT_COLS], on='spot_id', how='left'
)
for col in SUBMIT_COLS:
    submission[col] = submission[col].fillna('background')

out_path = SCRATCH / 'submission_phase2.csv'
submission.to_csv(str(out_path), index=False)

print(f'\n===== SUBMISSION SUMMARY =====')
print(f'  Rows:             {len(submission):,}')
bkg = (submission['class'] == 'background').mean()
print(f'  Background spots: {bkg:.1%}  (ground truth is ~83%)')
for col in SUBMIT_COLS:
    n_unique = submission[col].nunique()
    print(f'  Unique {col} labels: {n_unique}')
print(f'  Saved to: {out_path}')
print('\nDownload submission_phase2.csv and upload to Kaggle.')

## 9. Optional Improvements

The cells below implement optional improvements to try if the baseline submission
leaves room to improve. Run them in order and re-generate the submission.

In [ ]:
# Improvement A: Gradient Boosted Trees instead of Random Forest
# GBT typically outperforms RF on tabular gene expression data.
# Uses scikit-learn's HistGradientBoostingClassifier (fast, handles large datasets)

from sklearn.ensemble import HistGradientBoostingClassifier

classifiers_gbt = {level: {} for level in HIERARCHY}
encoders_gbt    = {level: {} for level in HIERARCHY}

# Level 0: class
level = HIERARCHY[0]
y     = adata.obs[level].values
le    = LabelEncoder().fit(y)
encoders_gbt[level]['root'] = le
clf   = HistGradientBoostingClassifier(
    max_iter=200, learning_rate=0.1, max_depth=6,
    random_state=42, class_weight='balanced'
)
clf.fit(X_features, le.transform(y))
classifiers_gbt[level]['root'] = clf
train_preds_gbt = le.inverse_transform(clf.predict(X_features))
print(f'GBT Level 0 ({level}): train ARI={adjusted_rand_score(y, train_preds_gbt):.4f}')

# Levels 1-3 (same hierarchy approach)
parent_preds_gbt = {HIERARCHY[0]: train_preds_gbt}
for i in range(1, 4):
    level      = HIERARCHY[i]
    parent_lvl = HIERARCHY[i-1]
    parent_p   = parent_preds_gbt[parent_lvl]
    y_curr     = adata.obs[level].values
    curr_preds = np.full(len(y_curr), 'background', dtype=object)

    for parent_class in np.unique(parent_p):
        idx = np.where(parent_p == parent_class)[0]
        if len(idx) < 5: continue
        X_sub = X_features[idx]
        y_sub = y_curr[idx]
        unique_labels = np.unique(y_sub)
        if len(unique_labels) == 1:
            curr_preds[idx] = unique_labels[0]
            classifiers_gbt[level][parent_class] = None
            encoders_gbt[level][parent_class] = unique_labels[0]
            continue
        le  = LabelEncoder().fit(y_sub)
        clf = HistGradientBoostingClassifier(
            max_iter=200, learning_rate=0.1, max_depth=6,
            random_state=42, class_weight='balanced'
        )
        clf.fit(X_sub, le.transform(y_sub))
        curr_preds[idx] = le.inverse_transform(clf.predict(X_sub))
        classifiers_gbt[level][parent_class] = clf
        encoders_gbt[level][parent_class]    = le

    ari = adjusted_rand_score(y_curr, curr_preds)
    print(f'GBT Level {i} ({level}): train ARI={ari:.4f}')
    parent_preds_gbt[level] = curr_preds

# To use GBT for submission, replace classifiers/encoders:
# classifiers = classifiers_gbt
# encoders = encoders_gbt
# Then re-run section 8
print('\nTo use GBT, uncomment the last two lines and re-run section 8.')

In [ ]:
# Improvement B: Spatial neighborhood features
# For each cell, add the mean expression of its k nearest neighbors
# as additional features. Cell types are spatially structured,
# so neighborhood context improves classification.

from sklearn.neighbors import NearestNeighbors

# Use cell centroids from adata.obs
centroid_cols = ['center_x', 'center_y']
if all(c in adata.obs.columns for c in centroid_cols):
    centroids = adata.obs[centroid_cols].values.astype(np.float32)

    # Find 10 nearest neighbors for each cell
    K = 10
    nbrs = NearestNeighbors(n_neighbors=K+1, algorithm='ball_tree').fit(centroids)
    _, indices = nbrs.kneighbors(centroids)
    indices = indices[:, 1:]  # exclude self

    # Mean neighbor expression (in PCA space)
    neighbor_features = np.stack([X_pca[indices[i]].mean(0)
                                   for i in range(len(X_pca))])
    X_with_neighbors  = np.concatenate([X_features, neighbor_features], axis=1)
    print(f'Features with neighborhood: {X_with_neighbors.shape}')
    print('To use: replace X_features with X_with_neighbors and retrain section 6.')
else:
    print('Centroid columns not found in adata.obs — skipping neighborhood features.')
    print(f'Available columns: {adata.obs.columns.tolist()}')

## Notes for report

**Pipeline:**
1. Segmentation: pretrained Cellpose v4, DAPI only, z=2, diameter=20
2. Expression: per-cell gene count vectors from spots inside each segmented cell, ~1,147 genes
3. Normalization: library-size normalization then log1p, PCA to 50 components
4. Classification: hierarchy-aware Random Forest — separate model per hierarchy level, conditioned on parent-level prediction
5. Background: spots outside any segmented cell → `background` at all 4 levels (~83% of spots)

**Why hierarchy-aware beats flat kNN:**
- Each sub-model only needs to distinguish among labels that share the same parent, reducing the classification problem size
- Errors at coarse levels don't propagate confusion across unrelated branches
- The flat kNN baseline treats all 10+ labels equally at every level

**Limitations:**
- Segmentation quality is the ceiling — misassigned spots can't be recovered by the classifier
- CCF coordinates are not available for test cells, so spatial features are approximate
- Expression vectors are noisy because each cell has only ~50–200 detected spots across 1,147 genes